# 08. 합성곱 신경망 (Convolutional Neural Network)

이번 장에서는 이미지 처리에 탁월한 성능을 보이는 **합성곱 신경망(Convolutional Neural Network, CNN)**을 다룬다. 앞 장에서 본 다층 퍼셉트론(MLP)이 이미지를 다룰 때 어떤 한계가 있는지부터 짚고, 합성곱 연산과 풀링 연산의 의미를 단계별로 정리한 뒤, CNN으로 MNIST를 분류하는 실습까지 진행한다.

> **이 장의 흐름** : MLP의 한계 → 채널 개념 → 합성곱 연산(커널·스트라이드·패딩) → 가중치 공유와 편향 → 특성 맵 크기 계산 → 다채널/다중 커널 합성곱 → 풀링 → CNN으로 MNIST 분류 → 더 깊은 CNN으로 MNIST 분류.
>
> 학습의 골격은 앞 장들과 같다. **순전파로 예측 → 손실 함수로 오차 계산 → 역전파로 기울기 계산 → 옵티마이저로 가중치 갱신**을 반복한다. 달라지는 것은 입력의 공간 구조를 보존하며 특징을 추출하는 **합성곱층**이라는 새로운 구조뿐이다.

# 1. 합성곱과 풀링 (Convolution and Pooling)

합성곱 신경망은 크게 **합성곱층(Convolution layer)**과 **풀링층(Pooling layer)**으로 구성된다. 일반적인 CNN은 `합성곱 → 활성화 함수(ReLU) → 풀링`이 반복되다가, 마지막에 전결합층(Fully-Connected layer)으로 분류를 수행하는 형태를 띤다.

이 절에서는 합성곱 연산과 풀링 연산이 각각 무엇을 하는지부터 차근차근 이해한다.

## 1-1. 합성곱 신경망의 대두

이미지 처리를 위해 앞서 배운 다층 퍼셉트론을 쓸 수도 있지만 한계가 있다. 예를 들어 알파벳 손글씨 `Y`를 비교적 정자로 쓴 것과 다소 휘갈겨 쓴 것을 행렬로 표현하면, 사람이 보기에는 둘 다 같은 `Y`지만 기계가 보기에는 각 픽셀값이 거의 상이하므로 완전히 다른 입력이 된다. 이미지는 같은 대상이라도 휘어지거나, 이동되거나, 방향이 뒤틀리는 등 다양한 변형이 존재하는데, **다층 퍼셉트론은 몇 개의 픽셀 값만 달라져도 민감하게 예측에 영향을 받는다**는 단점이 있다.

좀 더 구체적으로, 다층 퍼셉트론으로 손글씨를 분류하려면 2차원 이미지를 1차원 벡터로 펼쳐서 입력층으로 사용해야 한다. 그런데 이렇게 1차원으로 변환된 결과는 사람이 보기에도 원래 어떤 이미지였는지 알아보기 어렵다. 즉 변환 전에 가지고 있던 **공간적 구조(spatial structure) 정보가 유실**된 상태다.

> 공간적 구조 정보란 거리가 가까운 픽셀들끼리는 어떤 연관이 있고, 어떤 픽셀들끼리는 값이 비슷하다는 등의 정보를 포함한다. 결국 이미지의 공간적 구조 정보를 **보존하면서** 학습할 수 있는 방법이 필요해졌고, 이를 위해 사용하는 것이 합성곱 신경망이다.

## 1-2. 채널 (Channel)

기계는 글자나 이미지보다 숫자, 즉 텐서를 더 잘 처리한다. 이미지는 **(높이, 너비, 채널)**이라는 3차원 텐서다.

- **높이(height)** : 이미지의 세로 방향 픽셀 수
- **너비(width)** : 이미지의 가로 방향 픽셀 수
- **채널(channel)** : 색 성분. 깊이(depth)라고도 한다.

흑백 이미지는 채널 수가 1이며 각 픽셀은 0부터 255 사이의 값을 가진다. 따라서 $28 \times 28$ 흑백 손글씨 이미지는 $(28 \times 28 \times 1)$ 크기의 3차원 텐서다. 반면 우리가 통상적으로 접하는 컬러 이미지는 적색(Red)·녹색(Green)·청색(Blue) 세 가지 채널을 가지므로, $28 \times 28$ 컬러 이미지는 $(28 \times 28 \times 3)$ 크기의 3차원 텐서가 된다.

## 1-3. 합성곱 연산 (Convolution operation)

합성곱층은 **합성곱 연산을 통해 이미지의 특징을 추출**하는 역할을 한다. 합성곱은 영어로 컨볼루션이라고도 불리며, **커널(kernel)** 또는 **필터(filter)**라는 $n \times m$ 크기의 행렬로 높이$\times$너비 크기의 이미지를 처음부터 끝까지 겹치며 훑으면서, 겹쳐지는 부분의 이미지 원소와 커널 원소를 곱해 모두 더한 값을 출력으로 하는 연산이다. 이때 이미지의 가장 왼쪽 위부터 오른쪽 아래까지 순차적으로 훑는다.

> 커널은 일반적으로 $3 \times 3$ 또는 $5 \times 5$ 크기를 사용한다.

예를 들어 $3 \times 3$ 커널로 $5 \times 5$ 이미지에 합성곱 연산을 수행하는 첫 네 스텝(step)은 다음과 같이 계산된다(한 번의 연산을 1 스텝이라 한다).

$$ \text{1스텝} : (1 \times 1) + (2 \times 0) + (3 \times 1) + (2 \times 1) + (1 \times 0) + (0 \times 1) + (3 \times 0) + (0 \times 1) + (1 \times 0) = 6 $$
$$ \text{2스텝} : (2 \times 1) + (3 \times 0) + (4 \times 1) + (1 \times 1) + (0 \times 0) + (1 \times 1) + (0 \times 0) + (1 \times 1) + (1 \times 0) = 9 $$
$$ \text{3스텝} : (3 \times 1) + (4 \times 0) + (5 \times 1) + (0 \times 1) + (1 \times 0) + (2 \times 1) + (1 \times 0) + (1 \times 1) + (0 \times 0) = 11 $$
$$ \text{4스텝} : (2 \times 1) + (1 \times 0) + (0 \times 1) + (3 \times 1) + (0 \times 0) + (1 \times 1) + (1 \times 0) + (4 \times 1) + (1 \times 0) = 10 $$

이렇게 입력으로부터 커널을 사용해 합성곱 연산을 통해 나온 결과를 **특성 맵(feature map)**이라 한다.

커널의 이동 범위 또한 사용자가 정할 수 있는데, 이 이동 범위를 **스트라이드(stride)**라 한다. 위 예제는 스트라이드가 1이지만, 스트라이드를 2로 하면 한 번에 두 칸씩 이동하므로 더 작은 특성 맵을 얻는다.

## 1-4. 패딩 (Padding)

합성곱 연산의 결과로 얻은 특성 맵은 입력보다 크기가 작아진다는 특징이 있다. $5 \times 5$ 이미지에 $3 \times 3$ 커널을 스트라이드 1로 합성곱하면 $3 \times 3$ 특성 맵이 된다. 만약 합성곱층을 여러 개 쌓으면 최종 특성 맵은 초기 입력보다 매우 작아져 버린다. 합성곱 연산 이후에도 특성 맵의 크기를 입력과 동일하게 유지하고 싶다면 **패딩(padding)**을 사용한다.

패딩은 합성곱 연산을 하기 전에 입력의 가장자리에 지정된 폭만큼 행과 열을 추가하는 것이다. 주로 값을 0으로 채우는 **제로 패딩(zero padding)**을 사용한다.

> 스트라이드가 1일 때, $3 \times 3$ 커널을 쓴다면 1폭짜리 제로 패딩을, $5 \times 5$ 커널을 쓴다면 2폭짜리 제로 패딩을 쓰면 입력과 특성 맵의 크기를 보존할 수 있다. 예를 들어 $5 \times 5$ 이미지에 1폭 제로 패딩을 하면 $7 \times 7$ 이 되고, 여기에 $3 \times 3$ 커널을 스트라이드 1로 합성곱하면 다시 $5 \times 5$ 특성 맵이 된다.

## 1-5. 가중치와 편향

### 합성곱 신경망의 가중치

다층 퍼셉트론과 비교해 보자. $3 \times 3$ 이미지를 다층 퍼셉트론으로 처리하려면 이미지를 1차원 벡터(원소 9개)로 만들어 입력층 뉴런 9개로 사용해야 한다. 여기에 뉴런 4개짜리 은닉층을 붙이면 연결선(가중치)은 $9 \times 4 = 36$ 개가 된다.

이제 같은 $3 \times 3$ 이미지를 $2 \times 2$ 커널, 스트라이드 1로 합성곱 신경망에서 처리한다고 하자. **합성곱 신경망에서 가중치는 커널 행렬의 원소들**이다. 최종적으로 특성 맵을 얻기 위해서는 동일한 커널로 이미지 전체를 훑으며 합성곱을 진행하므로, 이미지 전체에서 사용되는 가중치는 $w_0, w_1, w_2, w_3$ 단 4개뿐이다. 또한 각 합성곱 연산마다 이미지의 모든 픽셀이 아니라 커널과 맵핑되는 픽셀만 입력으로 사용한다.

> 결국 합성곱 신경망은 다층 퍼셉트론보다 **훨씬 적은 수의 가중치**를 사용하며, **공간적 구조 정보를 보존**한다는 특징이 있다(가중치 공유, weight sharing).

다층 퍼셉트론의 은닉층에서 가중치 연산 후 비선형성을 위해 활성화 함수를 통과시켰던 것처럼, 합성곱 신경망에서도 특성 맵은 활성화 함수를 지난다. 이때 주로 ReLU 함수나 그 변형이 사용된다. 이렇게 합성곱 연산으로 특성 맵을 얻고 활성화 함수를 지나는 연산을 하는 은닉층을 **합성곱 층(convolution layer)**이라 한다.

### 합성곱 신경망의 편향

합성곱 신경망에도 **편향(bias)**을 추가할 수 있다. 편향을 사용한다면 커널을 적용한 뒤에 더해진다. 편향은 **하나의 값**만 존재하며, 커널이 적용된 결과의 모든 원소에 동일하게 더해진다.

## 1-6. 특성 맵의 크기 계산 방법

입력의 크기와 커널의 크기, 스트라이드 값만 알면 합성곱 연산 결과인 특성 맵의 크기를 계산할 수 있다. 각 변수는 다음을 의미한다.

- $I_h$ : 입력의 높이,  $I_w$ : 입력의 너비
- $K_h$ : 커널의 높이,  $K_w$ : 커널의 너비
- $S$ : 스트라이드
- $O_h$ : 특성 맵의 높이,  $O_w$ : 특성 맵의 너비

이때 특성 맵의 높이와 너비는 다음과 같다.

$$ O_h = floor\left(\frac{I_h - K_h}{S} + 1\right) $$
$$ O_w = floor\left(\frac{I_w - K_w}{S} + 1\right) $$

여기서 $floor$ 함수는 소수점 이하를 버린다. 예를 들어 $5 \times 5$ 이미지에 $3 \times 3$ 커널을 스트라이드 1로 합성곱하면 특성 맵의 크기는 $(5 - 3 + 1) \times (5 - 3 + 1) = 3 \times 3$ 이며, 이는 총 9번의 스텝이 필요함을 의미한다.

패딩의 폭을 $P$ 라 하면, 패딩까지 고려한 식은 다음과 같다.

$$ O_h = floor\left(\frac{I_h - K_h + 2P}{S} + 1\right) $$
$$ O_w = floor\left(\frac{I_w - K_w + 2P}{S} + 1\right) $$

이 공식을 코드로 직접 확인해 본다.

In [ ]:
import math

# 특성 맵 한 변의 크기를 계산하는 함수
def conv_output_size(input_size, kernel_size, stride=1, padding=0):
    return math.floor((input_size - kernel_size + 2 * padding) / stride + 1)

# 5x5 입력, 3x3 커널, 스트라이드 1, 패딩 0 -> 3
print('5x5, 3x3 커널, s=1, p=0 :', conv_output_size(5, 3, stride=1, padding=0))
# 28x28 입력, 3x3 커널, 스트라이드 1, 패딩 1 -> 28 (크기 보존)
print('28x28, 3x3 커널, s=1, p=1 :', conv_output_size(28, 3, stride=1, padding=1))
# 5x5 입력, 3x3 커널, 스트라이드 2, 패딩 0 -> 2
print('5x5, 3x3 커널, s=2, p=0 :', conv_output_size(5, 3, stride=2, padding=0))

## 1-7. 다수의 채널을 가질 경우의 합성곱 연산

지금까지는 채널(깊이)을 고려하지 않고 2차원 텐서를 가정했다. 하지만 실제 합성곱 연산의 입력은 다수의 채널을 가진 이미지(예: RGB)이거나 이전 연산의 결과로 나온 특성 맵일 수 있다.

다수의 채널을 가진 입력에 합성곱을 한다면, **커널의 채널 수도 입력의 채널 수만큼 존재해야 한다.** 즉 입력 데이터의 채널 수와 커널의 채널 수는 같아야 한다. 채널 수가 같으므로 합성곱 연산을 채널마다 수행하고, 그 결과를 **모두 더해서** 하나의 채널을 가지는 특성 맵을 만든다.

> 주의할 점은 이 연산에서 사용되는 커널이 3개의 커널이 아니라 **3개의 채널을 가진 1개의 커널**이라는 점이다. 합성곱 결과로 얻은 특성 맵의 채널 차원은 RGB 같은 컬러의 의미를 담고 있지 않다.

예를 들어 (높이 3, 너비 3, 채널 3) 입력이 (높이 2, 너비 2, 채널 3) 커널과 합성곱하면 (높이 2, 너비 2, 채널 1) 특성 맵을 얻는다.

## 1-8. 3차원 텐서의 합성곱 연산과 다수의 커널

(높이 $I_h$, 너비 $I_w$, 채널 $C_i$) 입력은 동일한 채널 수 $C_i$ 를 가지는 (높이 $K_h$, 너비 $K_w$) 커널과 합성곱하여 (높이 $O_h$, 너비 $O_w$, 채널 1) 특성 맵을 얻는다. 그런데 하나의 입력에 **여러 개의 커널**을 사용할 수도 있다.

> 합성곱 연산에서 사용한 커널의 수($C_o$)는 곧 합성곱 결과로 나오는 **특성 맵의 채널 수**가 된다.

이를 이해했다면 가중치 매개변수의 총 개수를 구할 수 있다. 가중치는 커널의 원소들이므로, 하나의 커널의 하나의 채널은 $K_h \times K_w$ 개의 매개변수를 가진다. 그런데 커널은 입력 데이터의 채널 수 $C_i$ 만큼 채널을 가져야 하므로 하나의 커널이 가지는 매개변수의 수는 $K_h \times K_w \times C_i$ 다. 이러한 커널이 총 $C_o$ 개 있어야 하므로 가중치 매개변수의 총 수는 다음과 같다.

$$ \text{가중치 매개변수의 총 수} = K_h \times K_w \times C_i \times C_o $$

## 1-9. 풀링 (Pooling)

일반적으로 합성곱 층(합성곱 연산 + 활성화 함수) 다음에는 **풀링 층**을 추가한다. 풀링 층에서는 특성 맵을 다운샘플링하여 크기를 줄이는 풀링 연산이 이루어진다. 풀링 연산에는 일반적으로 **최대 풀링(max pooling)**과 **평균 풀링(average pooling)**이 사용된다.

- **최대 풀링** : 커널과 겹치는 영역 안에서 **최대값**을 추출하는 방식으로 다운샘플링한다.
- **평균 풀링** : 최대값 대신 **평균값**을 추출한다.

풀링 연산도 합성곱 연산처럼 커널과 스트라이드 개념을 가진다. 예를 들어 스트라이드 2, $2 \times 2$ 커널로 맥스 풀링을 하면 특성 맵이 절반 크기로 다운샘플링된다.

> 풀링 연산이 합성곱 연산과 다른 점은 두 가지다. ① **학습해야 할 가중치가 없다.** ② 연산 후에 **채널 수가 변하지 않는다.**

# 2. CNN으로 MNIST 분류하기

이제 앞에서 배운 합성곱과 풀링을 조합해 CNN으로 MNIST를 분류한다.

## 2-1. 모델 이해하기

합성곱 신경망은 출처에 따라 '합성곱 층'을 부르는 단위가 조금 다르다.

- **첫 번째 표기** : 합성곱(`nn.Conv2d`) + 활성화 함수(`nn.ReLU`)를 하나의 합성곱 층으로 보고, 맥스풀링(`nn.MaxPool2d`)은 풀링 층으로 별도로 명명한다.
- **두 번째 표기** : 합성곱 + 활성화 함수 + 맥스풀링까지를 하나의 합성곱 층으로 본다.

풀링을 하나의 층으로 보느냐 마느냐의 문제일 뿐 옳고 그름의 문제는 아니다. 이 장에서는 편의를 위해 **두 번째 표기**를 택한다. 그러면 우리가 만들 모델은 총 3개의 층으로 구성된다.

```text
# 1번 레이어 : 합성곱층
합성곱(in=1, out=32, kernel=3, stride=1, padding=1) + ReLU
맥스풀링(kernel=2, stride=2)

# 2번 레이어 : 합성곱층
합성곱(in=32, out=64, kernel=3, stride=1, padding=1) + ReLU
맥스풀링(kernel=2, stride=2)

# 3번 레이어 : 전결합층
특성 맵을 펼친다.  # batch_size x 7 x 7 x 64 -> batch_size x 3136
전결합층(뉴런 10개) + 활성화 함수 Softmax
```

## 2-2. 모델 구현하기 (텐서 크기 추적)

층을 곧바로 쌓기 전에, 임의의 텐서를 만들어 각 연산을 통과시키며 텐서의 크기가 어떻게 변하는지 직접 추적한다. 먼저 도구를 임포트하고, 크기가 $1 \times 1 \times 28 \times 28$ 인 텐서를 선언한다. 텐서의 형태는 **배치 크기 × 채널 × 높이 × 너비** 순서다.

In [ ]:
import torch
import torch.nn as nn

# 배치 1, 채널 1, 높이 28, 너비 28 크기의 텐서 선언
inputs = torch.Tensor(1, 1, 28, 28)
print('텐서의 크기 : {}'.format(inputs.shape))

합성곱층과 풀링을 선언한다. 첫 번째 합성곱은 1채널을 입력받아 32채널을 뽑아내고, 두 번째 합성곱은 32채널을 입력받아 64채널을 뽑아낸다. 커널 사이즈는 3, 패딩은 1이다. 맥스풀링은 정수 하나만 인자로 넣으면 커널 사이즈와 스트라이드가 모두 그 값으로 지정된다.

In [ ]:
conv1 = nn.Conv2d(1, 32, 3, padding=1)
print(conv1)

conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
print(conv2)

pool = nn.MaxPool2d(2)
print(pool)

이제 선언만 한 구현체들을 연결해 입력 텐서를 차례로 통과시키며 크기를 확인한다. 패딩을 1폭으로 하고 $3 \times 3$ 커널을 사용하면 합성곱 후에도 크기가 보존되고, 맥스풀링을 통과하면 높이와 너비가 절반이 된다.

In [ ]:
out = conv1(inputs)   # 1번 합성곱 통과
print(out.shape)      # 32채널, 28x28 (패딩 1 + 3x3 커널이라 크기 보존)

out = pool(out)       # 맥스풀링 통과
print(out.shape)      # 32채널, 14x14 (절반으로 다운샘플링)

out = conv2(out)      # 2번 합성곱 통과
print(out.shape)      # 64채널, 14x14

out = pool(out)       # 맥스풀링 통과
print(out.shape)      # 64채널, 7x7

`.size(n)` 으로 텐서의 $n$ 번째 차원에 접근할 수 있다. 현재 `out` 의 크기는 $1 \times 64 \times 7 \times 7$ 이다.

In [ ]:
print(out.size(0))  # 배치 차원 -> 1
print(out.size(1))  # 채널 -> 64
print(out.size(2))  # 높이 -> 7
print(out.size(3))  # 너비 -> 7

`.view()` 로 텐서를 펼친다. **첫 번째 차원인 배치 차원은 그대로 두고 나머지는 하나로 펼친다.** 그런 다음 전결합층(`nn.Linear`)을 통과시켜 10개 차원의 텐서로 변환한다($7 \times 7 \times 64 = 3136$).

In [ ]:
# 배치 차원은 두고 나머지를 펼친다
out = out.view(out.size(0), -1)
print(out.shape)  # [1, 3136]

fc = nn.Linear(3136, 10)  # input_dim=3136, output_dim=10
out = fc(out)
print(out.shape)  # [1, 10]

## 2-3. CNN으로 MNIST 분류하기

위에서 확인한 구조를 클래스로 묶어 실제로 MNIST를 분류한다. 먼저 필요한 도구를 임포트하고, GPU 사용이 가능하면 `device` 를 `cuda` 로, 아니면 `cpu` 로 설정한다. 결과 재현을 위해 랜덤 시드를 고정한다.

In [ ]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 랜덤 시드 고정
torch.manual_seed(777)

# GPU 사용 가능일 경우 랜덤 시드 고정
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

학습에 사용할 하이퍼파라미터를 설정한다.

In [ ]:
learning_rate = 0.001
training_epochs = 15
batch_size = 100

데이터셋을 정의하고 `DataLoader` 로 배치 크기를 지정한다. `transforms.ToTensor()` 는 0~255 픽셀값을 $[0, 1]$ 구간의 텐서로 변환한다. `drop_last=True` 는 마지막에 남는 배치 크기 미만의 데이터를 버려, 모든 배치의 크기를 동일하게 맞춘다.

In [ ]:
mnist_train = dsets.MNIST(root='MNIST_data/',          # 다운로드 경로 지정
                          train=True,                  # True면 훈련 데이터로 다운로드
                          transform=transforms.ToTensor(),  # 텐서로 변환
                          download=True)

mnist_test = dsets.MNIST(root='MNIST_data/',           # 다운로드 경로 지정
                         train=False,                  # False면 테스트 데이터로 다운로드
                         transform=transforms.ToTensor(),  # 텐서로 변환
                         download=True)

data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

이제 클래스로 모델을 설계한다. `torch.nn.Sequential` 로 각 층을 묶고, 전결합층 한정으로 세이비어 초기화를 적용한다. 각 층의 주석에 입력·출력 텐서의 형태를 함께 적어 두면 크기 흐름을 추적하기 좋다.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 첫 번째 층
        # ImgIn shape=(?, 1, 28, 28)
        #   Conv -> (?, 32, 28, 28)
        #   Pool -> (?, 32, 14, 14)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

        # 두 번째 층
        # ImgIn shape=(?, 32, 14, 14)
        #   Conv -> (?, 64, 14, 14)
        #   Pool -> (?, 64, 7, 7)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

        # 전결합층 7x7x64 inputs -> 10 outputs
        self.fc = torch.nn.Linear(7 * 7 * 64, 10, bias=True)
        # 전결합층 한정으로 가중치 초기화
        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)  # 전결합층을 위해 Flatten
        out = self.fc(out)
        return out

모델을 정의하고 비용 함수와 옵티마이저를 설정한다.

> `nn.CrossEntropyLoss` 에는 **소프트맥스 함수가 이미 포함**되어 있다. 그래서 모델의 마지막 `forward` 에서 따로 소프트맥스를 적용하지 않는다. 만약 로짓에 소프트맥스를 한 번 더 적용한 뒤 이 손실 함수에 넘기면 잘못된 학습이 되므로 주의한다.

In [ ]:
# CNN 모델 정의
model = CNN().to(device)

criterion = torch.nn.CrossEntropyLoss().to(device)  # 비용 함수에 소프트맥스 함수 포함
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

총 배치의 수를 출력한다. 배치 크기를 100으로 했으므로 총 배치 수가 600이면 훈련 데이터는 총 60,000개라는 의미다.

In [ ]:
total_batch = len(data_loader)
print('총 배치의 수 : {}'.format(total_batch))

이제 모델을 훈련시킨다. 미니 배치 단위로 데이터를 꺼내 `순전파 → 손실 계산 → 역전파 → 파라미터 갱신`을 반복한다. (시간이 꽤 오래 걸린다. Colab에서는 런타임을 GPU로 바꾸면 훨씬 빠르다.)

In [ ]:
for epoch in range(training_epochs):
    avg_cost = 0  # 에포크당 평균 비용을 저장할 변수 초기화

    for X, Y in data_loader:  # 미니 배치 단위로 꺼낸다. X는 입력, Y는 레이블.
        # 이미지 데이터는 이미 (28x28) 크기이므로 별도의 reshape 불필요
        # 레이블 Y는 원-핫 인코딩이 아닌 정수형 클래스 레이블
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()             # 옵티마이저의 기울기 초기화
        hypothesis = model(X)             # 순전파 연산으로 예측값 계산
        cost = criterion(hypothesis, Y)   # 예측값과 실제값 Y의 손실 계산
        cost.backward()                   # 역전파 연산으로 기울기 계산
        optimizer.step()                  # 옵티마이저로 파라미터 갱신

        avg_cost += cost / total_batch    # 현재 배치 비용을 전체 배치 수로 나누어 누적

    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

학습이 끝났으니 테스트 데이터로 정확도를 평가한다. 약 **98%**의 정확도가 나온다. `torch.no_grad()` 블록은 기울기 계산을 하지 않아 메모리 사용을 최적화한다.

> **deprecated API 주의** : 예전 교재 코드는 `mnist_test.test_data` / `mnist_test.test_labels` 를 사용했지만, 현재 torchvision에서는 이 속성이 제거되어 `mnist_test.data` / `mnist_test.targets` 를 사용해야 한다.
>
> 또한 학습 데이터는 `ToTensor()` 로 $[0, 1]$ 로 정규화된 반면, 아래의 `.data` 는 0~255 범위의 원본 픽셀값이다. 엄밀히는 테스트도 동일하게 정규화해야 분포가 일치한다(`.float() / 255.`). 교재 코드는 이를 생략하고도 높은 정확도를 얻지만, 실전에서는 훈련·추론의 전처리를 반드시 맞추는 것이 옳다.

In [ ]:
# 학습을 진행하지 않으므로 torch.no_grad() 사용
with torch.no_grad():
    # 테스트 데이터를 (배치, 채널, 높이, 너비) 형태로 변환하고 연산 장치로 이동
    X_test = mnist_test.data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.targets.to(device)

    prediction = model(X_test)  # 테스트 데이터에 대한 모델 예측
    correct_prediction = torch.argmax(prediction, 1) == Y_test  # 예측 클래스와 실제 레이블 일치 여부
    accuracy = correct_prediction.float().mean()  # 일치하는 예측의 평균 = 정확도
    print('Accuracy:', accuracy.item())

# 3. 깊은 CNN으로 MNIST 분류하기

앞서 만든 CNN에 층을 더 추가해 MNIST를 분류한다. 사실 이번 절의 코드는 이전 절에서 층이 조금 더 추가되는 것 말고는 동일하다.

## 3-1. 모델 이해하기

우리가 만들 모델은 총 5개의 층으로 구성된다. 앞 절의 1번·2번 레이어는 동일하되, 새로운 합성곱층과 전결합층을 추가했다.

```text
# 1번 레이어 : 합성곱층
합성곱(in=1, out=32, kernel=3, stride=1, padding=1) + ReLU
맥스풀링(kernel=2, stride=2)

# 2번 레이어 : 합성곱층
합성곱(in=32, out=64, kernel=3, stride=1, padding=1) + ReLU
맥스풀링(kernel=2, stride=2)

# 3번 레이어 : 합성곱층
합성곱(in=64, out=128, kernel=3, stride=1, padding=1) + ReLU
맥스풀링(kernel=2, stride=2, padding=1)

# 4번 레이어 : 전결합층
특성 맵을 펼친다.  # batch_size x 4 x 4 x 128 -> batch_size x 2048
전결합층(뉴런 625개) + ReLU + 드롭아웃(p=0.5)

# 5번 레이어 : 전결합층
전결합층(뉴런 10개) + 활성화 함수 Softmax
```

> 3번 레이어의 맥스풀링에는 패딩 1이 들어간다. $7 \times 7$ 특성 맵에 $3 \times 3$ 합성곱(패딩 1)을 하면 $7 \times 7$ 이 유지되고, 여기에 패딩 1짜리 $2 \times 2$ 맥스풀링(스트라이드 2)을 하면 $floor((7 + 2 \times 1 - 2)/2 + 1) = 4$ 가 되어 $4 \times 4$ 가 된다. 그래서 전결합층의 입력이 $4 \times 4 \times 128 = 2048$ 이다.

## 3-2. 깊은 CNN으로 MNIST 분류하기

도구 임포트, device 설정, 시드 고정, 하이퍼파라미터, 데이터 로드는 앞 절과 동일하다. 한 셀에 모아 실행한다.

In [ ]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 랜덤 시드 고정
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

learning_rate = 0.001
training_epochs = 15
batch_size = 100

mnist_train = dsets.MNIST(root='MNIST_data/',
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)

mnist_test = dsets.MNIST(root='MNIST_data/',
                         train=False,
                         transform=transforms.ToTensor(),
                         download=True)

data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

이제 5개 층을 가진 깊은 CNN 클래스를 설계한다. 4번 레이어(첫 전결합층)에는 과적합을 막기 위한 **드롭아웃**(`nn.Dropout`)이 들어간다. `keep_prob = 0.5` 이므로 드롭아웃 확률은 $1 - 0.5 = 0.5$ 다.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.keep_prob = 0.5  # 드롭아웃에서 유지할 비율

        # L1: 첫 번째 합성곱층
        #   입력 (?, 1, 28, 28) -> Conv (?, 32, 28, 28) -> Pool (?, 32, 14, 14)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

        # L2: 두 번째 합성곱층
        #   입력 (?, 32, 14, 14) -> Conv (?, 64, 14, 14) -> Pool (?, 64, 7, 7)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))

        # L3: 세 번째 합성곱층
        #   입력 (?, 64, 7, 7) -> Conv (?, 128, 7, 7) -> Pool (?, 128, 4, 4)
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=1))

        # L4: 첫 번째 전결합층 4x4x128 -> 625, ReLU, Dropout(p=0.5)
        self.fc1 = torch.nn.Linear(4 * 4 * 128, 625, bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight)  # 가중치 초기화
        self.layer4 = torch.nn.Sequential(
            self.fc1,
            torch.nn.ReLU(),
            torch.nn.Dropout(p=1 - self.keep_prob))

        # L5: 최종 전결합층 625 -> 10
        self.fc2 = torch.nn.Linear(625, 10, bias=True)
        torch.nn.init.xavier_uniform_(self.fc2.weight)  # 가중치 초기화

    def forward(self, x):
        out = self.layer1(x)             # 1번 합성곱층 통과
        out = self.layer2(out)           # 2번 합성곱층 통과
        out = self.layer3(out)           # 3번 합성곱층 통과
        out = out.view(out.size(0), -1)  # 전결합층 입력을 위해 Flatten
        out = self.layer4(out)           # 1번 전결합층 통과
        out = self.fc2(out)              # 최종 전결합층 통과
        return out

모델·비용 함수·옵티마이저를 정의하고 총 배치 수를 확인한다.

> 드롭아웃이 포함되어 있으므로 학습 시에는 `model.train()`, 추론 시에는 `model.eval()` 로 전환하는 것이 중요하다. `eval()` 모드에서는 드롭아웃이 자동으로 비활성화된다.

In [ ]:
# CNN 모델 정의
model = CNN().to(device)

criterion = torch.nn.CrossEntropyLoss().to(device)  # 비용 함수에 소프트맥스 함수 포함
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

total_batch = len(data_loader)
print('총 배치의 수 : {}'.format(total_batch))

모델을 훈련시킨다.

In [ ]:
model.train()  # 드롭아웃을 사용하기 위해 학습 모드로 전환

for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:  # 미니 배치 단위로 꺼낸다. X는 입력, Y는 레이블.
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()
        hypothesis = model(X)
        cost = criterion(hypothesis, Y)
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch

    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

테스트 데이터로 평가한다. (deprecated API 교정과 정규화 주의는 2-3절과 동일하다.)

In [ ]:
# 학습을 진행하지 않으므로 torch.no_grad() 사용
with torch.no_grad():
    model.eval()  # 추론 모드로 전환 (드롭아웃 비활성화)

    X_test = mnist_test.data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.targets.to(device)

    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())

층을 더 깊게 쌓았는데 오히려 정확도가 줄어드는 것을 볼 수 있다.

> **결론** : 층을 깊게 쌓는 것도 중요하지만, 깊게 쌓는 것이 **항상** 정확도를 올려 주지는 않는다. 효율적으로 쌓는 것도 그만큼 중요하다는 의미다. 무작정 깊이만 늘리기보다, 데이터의 성격과 과적합 여부를 보며 구조를 설계해야 한다.

---

이로써 8장 **합성곱 신경망**의 핵심을 모두 정리했다. 합성곱 연산과 풀링 연산이 공간 구조를 보존하며 특징을 추출하는 원리를 단계별로 살펴봤고, 특성 맵의 크기 계산과 가중치 공유의 이점을 확인했으며, CNN과 더 깊은 CNN으로 MNIST를 분류하는 실습까지 마쳤다. 다음 장부터는 텍스트를 다루는 **자연어 처리(NLP)**의 기초로 넘어간다.